- Web scraping
- `pip install beautifulsoup4 requests`


In [19]:
import requests
from bs4 import BeautifulSoup

In [ ]:
BASE_URL = "https://webscraper.io/test-sites/e-comerce/allinone//"

response = requests.get(BASE_URL)
# print("Status:", response.status_code)
# print("Content-Type:", response.headers['Content-Type'])

In [21]:
# print(response.text)

In [22]:
soup = BeautifulSoup(response.text, "html.parser")

# # Tip: Use .prettify() to inspect a small section
# sample = soup.find("div", class_="thumbnail")
# print(sample.prettify())

In [34]:
# Select all product cards on the homepage
cards = soup.select("div.thumbnail")
print(f"Found {len(cards)} product cards")


for card in cards:
    name  = card.select_one("a.title")["title"]           # read attribute, not text
    price = card.select_one("h4.price").get_text(strip=True)        # count filled stars
    stars = len(card.select("span.ws-icon-star"))        # count filled stars
    print(f"  {name:45s}  {price:10s}  stars={stars}")

Found 3 product cards
  Asus EeeBook R416NA-FA014T                     $433.3      stars=1
  Iphone                                         $899.99     stars=1
  Acer Spin 5                                    $564.98     stars=2


In [35]:
for card in cards:
    link = card.select_one("a.title")["href"]        # relative URL
    img  = card.select_one("img")["src"]             # image source
    full_link = "https://webscraper.io" + link
    print(f"Product : {full_link}")
    print(f"Image   : {img}\n")

Product : https://webscraper.io/test-sites/e-commerce/allinone/product/77
Image   : /images/test-sites/e-commerce/items/cart2.png

Product : https://webscraper.io/test-sites/e-commerce/allinone/product/9
Image   : /images/test-sites/e-commerce/items/cart2.png

Product : https://webscraper.io/test-sites/e-commerce/allinone/product/47
Image   : /images/test-sites/e-commerce/items/cart2.png



In [36]:
def scrape_category(category_url):
    """Scrape all pages of a category and return a list of product dicts."""
    products = []
    url = category_url

    while url:
        resp = requests.get(url)
        s    = BeautifulSoup(resp.text, "html.parser")

        for card in s.select("div.thumbnail"):
            products.append({
                "name" : card.select_one("a.title")["title"],
                "price": card.select_one("h4.price").get_text(strip=True),
                "desc" : card.select_one("p.description").get_text(strip=True),
                "url"  : "https://webscraper.io" + card.select_one("a.title")["href"],
            })

        # Follow pagination
        next_btn = s.select_one("a[rel='next']")
        url = ("https://webscraper.io" + next_btn["href"]) if next_btn else None

    return products


phones = scrape_category(BASE_URL + "/phones")
print(f"Scraped {len(phones)} phones")
phones[:3]

Scraped 3 phones


[{'name': 'Samsung Galaxy',
  'price': '$93.99',
  'desc': '5 mpx. Android 5.0',
  'url': 'https://webscraper.io/test-sites/e-commerce/allinone/product/3'},
 {'name': 'Iphone',
  'price': '$899.99',
  'desc': 'White',
  'url': 'https://webscraper.io/test-sites/e-commerce/allinone/product/7'},
 {'name': 'Sony Xperia',
  'price': '$118.99',
  'desc': 'GPS, waterproof',
  'url': 'https://webscraper.io/test-sites/e-commerce/allinone/product/5'}]

In [ ]:
import csv

# Write to CSV
with open("result.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["names", "price", "desc", "url"])
    writer.writeheader()
    writer.writerows(phones)

print(f"Saved {len(phones)} rows to result.csv")

# Simple stats without pandas
prices = [float(p["price"].replace("$", "")) for p in phones]
avg = sum(prices) / len(prices)
print(f"\nAverage price: ${avg:.2f}")

Saved 3 rows to result.csv

Average price: $370.99


In [38]:
import time

HEADERS = {"User-Agent": "Mozilla/5.0 (educational scraper demo)"}

def polite_get(url, delay=1.0):
    """GET with a user-agent header and a small delay."""
    time.sleep(delay)
    return requests.get(url, headers=HEADERS)

# Scrape a single product page
product_url = phones[0]["url"]
resp = polite_get(product_url)
ps   = BeautifulSoup(resp.text, "html.parser")

print("Product:", ps.select_one("h1").get_text(strip=True))
print("Price  :", ps.select_one("h4.price").get_text(strip=True))
print("Reviews:", ps.select_one("div.ratings").get_text(strip=True))

Product: Test Sites
Price  : $93.99
Reviews: 3reviews


In [ ]:
def safe_text(tag, selector, default="N/A"):
    el = tag.select_one(selector)
    return el.get_text(strip=True) if el else default

# Scrape laptops with safe extraction
resp  = requests.get(BASE_URL + "/computers/laptops")
soup2 = BeautifulSoup(resp.text, "html.parser")

for card in soup2.select("div.thumbnail")[:5]:
    name  = safe_text(card, "a.title")
    price = safe_text(card, "h4.price")
    desc  = safe_text(card, "p.description")
    print(f"{name} | {price} | {desc[:60]}...")